# 03. Feature Engineering

This notebook converts the findings from exploratory data analysis into reproducible dataset versions for model training and comparison.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

## Environment Setup

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

Project root configured.
Processed directory configured.


In [3]:
from src.churn_ml.data import load_competition_data
from src.churn_ml.features import (
    PreparedDataset,
    add_missingness_indicator_features,
    add_missingness_summary_features,
    add_selected_missing_indicators,
    save_dataset,
)

## Metadata Loading

In [4]:
metadata_dir = PROJECT_ROOT / "data/interim/eda_metadata"


def load_feature_metadata(filename: str) -> list[str]:
    with (metadata_dir / filename).open("r", encoding="utf-8") as file:
        return json.load(file)


numeric_features = load_feature_metadata("numeric_features.json")
categorical_features = load_feature_metadata("categorical_features.json")
constant_features = load_feature_metadata("constant_features.json")
all_missing_features = load_feature_metadata("all_missing_features.json")
very_high_missing_features = load_feature_metadata("very_high_missing_features.json")

print(f"Numeric features:              {len(numeric_features)}")
print(f"Categorical features:          {len(categorical_features)}")
print(f"Constant features:             {len(constant_features)}")
print(f"All-missing features:          {len(all_missing_features)}")
print(f"Very high-missing features:    {len(very_high_missing_features)}")

Numeric features:              192
Categorical features:          38
Constant features:             25
All-missing features:          18
Very high-missing features:    136


## Dataset Loading

In [5]:
data = load_competition_data(
    train_path=PROJECT_ROOT / "data/raw/final_proj_data.csv",
    test_path=PROJECT_ROOT / "data/raw/final_proj_test.csv",
    sample_submission_path=(PROJECT_ROOT / "data/raw/final_proj_sample_submission.csv"),
)

print(f"Train features: {data.X.shape}")
print(f"Target:         {data.y.shape}")
print(f"Test features:  {data.test.shape}")

Train features: (10000, 230)
Target:         (10000,)
Test features:  (2500, 230)


## Data & Metadata Validation

In [6]:
train_columns = set(data.X.columns)

metadata_sets = {
    "numeric_features": set(numeric_features),
    "categorical_features": set(categorical_features),
    "constant_features": set(constant_features),
    "all_missing_features": set(all_missing_features),
    "very_high_missing_features": set(very_high_missing_features),
}

for name, features in metadata_sets.items():
    missing = features - train_columns
    extra = (
        train_columns - features
        if name
        in {
            "numeric_features",
            "categorical_features",
        }
        else set()
    )

    print(f"\n{name}")

    if missing:
        print(f"  Missing in dataset: {len(missing)}")
    else:
        print("  ✓ All metadata features exist in dataset")

    if extra:
        print(f"  Unclassified features: {len(extra)}")


numeric_features
  ✓ All metadata features exist in dataset
  Unclassified features: 38

categorical_features
  ✓ All metadata features exist in dataset
  Unclassified features: 192

constant_features
  ✓ All metadata features exist in dataset

all_missing_features
  ✓ All metadata features exist in dataset

very_high_missing_features
  ✓ All metadata features exist in dataset


## Dataset Version v0: Raw Minimal

In [7]:
features_to_drop = sorted(set(constant_features) | set(all_missing_features))

X_train_v0 = data.X.drop(columns=features_to_drop).copy()
X_test_v0 = data.test.drop(columns=features_to_drop).copy()

dataset_v0 = PreparedDataset(
    version="v0_raw_minimal",
    X_train=X_train_v0,
    y_train=data.y.copy(),
    X_test=X_test_v0,
    metadata={
        "description": (
            "Raw feature set with constant and all-missing features removed."
        ),
        "source": "raw",
        "dropped_features": features_to_drop,
        "n_dropped_features": len(features_to_drop),
        "n_train_rows": len(X_train_v0),
        "n_test_rows": len(X_test_v0),
        "n_features": X_train_v0.shape[1],
    },
)

print(f"Dataset version:   {dataset_v0.version}")
print(f"Train shape:       {dataset_v0.X_train.shape}")
print(f"Target shape:      {dataset_v0.y_train.shape}")
print(f"Test shape:        {dataset_v0.X_test.shape}")
print(f"Dropped features:  {dataset_v0.metadata['n_dropped_features']}")

Dataset version:   v0_raw_minimal
Train shape:       (10000, 205)
Target shape:      (10000,)
Test shape:        (2500, 205)
Dropped features:  25


In [8]:
saved_path = save_dataset(
    dataset=dataset_v0, processed_dir=PROCESSED_DIR, overwrite=True
)

print(f"Saved to: {saved_path}")

Saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\data\processed\v0_raw_minimal


## Dataset Version v1: Missingness Summary

This version retains all features from `v0_raw_minimal` and adds row-level summaries of missing values.

The hypothesis is that patterns of unavailable information may contain useful information about customer type, service usage, or data-collection processes.

In [9]:
X_train_v1 = add_missingness_summary_features(
    X_train_v0,
    very_high_missing_features=very_high_missing_features,
)

X_test_v1 = add_missingness_summary_features(
    X_test_v0,
    very_high_missing_features=very_high_missing_features,
)

added_features_v1 = [
    column
    for column in X_train_v1.columns
    if column not in X_train_v0.columns
]

if list(X_train_v1.columns) != list(X_test_v1.columns):
    raise ValueError(
        "Train and test columns differ after missingness "
        "feature generation."
    )

if X_train_v1[added_features_v1].isna().any().any():
    raise ValueError(
        "Generated train missingness features contain missing values."
    )

if X_test_v1[added_features_v1].isna().any().any():
    raise ValueError(
        "Generated test missingness features contain missing values."
    )

dataset_v1 = PreparedDataset(
    version="v1_missingness_summary",
    X_train=X_train_v1,
    y_train=data.y.copy(),
    X_test=X_test_v1,
    metadata={
        "description": (
            "v0_raw_minimal with row-level missing-value "
            "summary features."
        ),
        "source": "v0_raw_minimal",
        "base_version": "v0_raw_minimal",
        "added_features": added_features_v1,
        "n_added_features": len(added_features_v1),
        "n_train_rows": len(X_train_v1),
        "n_test_rows": len(X_test_v1),
        "n_features": X_train_v1.shape[1],
    },
)

print(f"Dataset version:   {dataset_v1.version}")
print(f"Train shape:       {dataset_v1.X_train.shape}")
print(f"Test shape:        {dataset_v1.X_test.shape}")
print(f"Added features:    {len(added_features_v1)}")
print(*added_features_v1, sep="\n")

Dataset version:   v1_missingness_summary
Train shape:       (10000, 213)
Test shape:        (2500, 213)
Added features:    8
missing_count_total
missing_rate_total
missing_count_numeric
missing_rate_numeric
missing_count_categorical
missing_rate_categorical
missing_count_very_high
missing_rate_very_high


In [10]:
display(
    X_train_v1[added_features_v1]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
missing_count_total,10000.0,135.467700,4.947242,128.000000,132.000000,135.000000,137.000000,157.000000
missing_rate_total,10000.0,0.660818,0.024133,0.624390,0.643902,0.658537,0.668293,0.765854
missing_count_numeric,10000.0,131.530800,3.168889,128.000000,130.000000,131.000000,132.000000,148.000000
missing_rate_numeric,10000.0,0.769186,0.018532,0.748538,0.760234,0.766082,0.771930,0.865497
missing_count_categorical,10000.0,3.936900,2.850076,0.000000,2.000000,4.000000,6.000000,12.000000
missing_rate_categorical,10000.0,0.115791,0.083826,0.000000,0.058824,0.117647,0.176471,0.352941
missing_count_very_high,10000.0,126.343200,8.201155,94.000000,129.000000,129.000000,129.000000,129.000000
missing_rate_very_high,10000.0,0.979405,0.063575,0.728682,1.000000,1.000000,1.000000,1.000000


In [11]:
saved_path_v1 = save_dataset(
    dataset=dataset_v1,
    processed_dir=PROCESSED_DIR,
    overwrite=True,
)

print(f"Saved to: {saved_path_v1}")

Saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\data\processed\v1_missingness_summary


## Dataset Version v2: Missingness Indicators

This version is built directly from `v0_raw_minimal`.

It adds:

- four row-level missing-value counts
- one binary missing-value indicator for every original feature that contains at least one missing value in the training dataset

The feature list is determined using training data only and is then applied unchanged to the test dataset.

In [12]:
missing_indicator_features_v2 = [
    column
    for column in X_train_v0.columns
    if X_train_v0[column].isna().any()
]

print(
    "Features with missing values in train: "
    f"{len(missing_indicator_features_v2)}"
)

print(
    "Expected new features: "
    f"{4 + len(missing_indicator_features_v2)}"
)

Features with missing values in train: 184
Expected new features: 188


In [13]:
X_train_v2 = add_missingness_indicator_features(
    X_train_v0,
    indicator_features=missing_indicator_features_v2,
    very_high_missing_features=very_high_missing_features,
)

X_test_v2 = add_missingness_indicator_features(
    X_test_v0,
    indicator_features=missing_indicator_features_v2,
    very_high_missing_features=very_high_missing_features,
)

added_features_v2 = [
    column
    for column in X_train_v2.columns
    if column not in X_train_v0.columns
]

indicator_columns_v2 = [
    f"{column}_is_missing"
    for column in missing_indicator_features_v2
]

summary_count_columns_v2 = [
    "missing_count_total",
    "missing_count_numeric",
    "missing_count_categorical",
    "missing_count_very_high",
]

c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\src\churn_ml\features.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[indicator_column] = dataframe[column].isna().astype("int8")
c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\src\churn_ml\features.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[indicator_column] = dataframe[column].isna().astype("int8")
c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\src\churn_m

In [14]:
if list(X_train_v2.columns) != list(X_test_v2.columns):
    raise ValueError(
        "Train and test columns differ after v2 feature generation."
    )

if X_train_v2.columns.duplicated().any():
    raise ValueError(
        "Generated train dataset contains duplicate columns."
    )

if X_train_v2[added_features_v2].isna().any().any():
    raise ValueError(
        "Generated train features contain missing values."
    )

if X_test_v2[added_features_v2].isna().any().any():
    raise ValueError(
        "Generated test features contain missing values."
    )

train_indicator_values = set(
    X_train_v2[indicator_columns_v2]
    .to_numpy()
    .ravel()
)

test_indicator_values = set(
    X_test_v2[indicator_columns_v2]
    .to_numpy()
    .ravel()
)

if not train_indicator_values.issubset({0, 1}):
    raise ValueError(
        "Train missing indicators contain values other than 0 and 1."
    )

if not test_indicator_values.issubset({0, 1}):
    raise ValueError(
        "Test missing indicators contain values other than 0 and 1."
    )

pd.testing.assert_frame_equal(
    X_train_v2[X_train_v0.columns],
    X_train_v0,
)

pd.testing.assert_frame_equal(
    X_test_v2[X_test_v0.columns],
    X_test_v0,
)

print("v2 validation passed.")

v2 validation passed.


In [15]:
dataset_v2 = PreparedDataset(
    version="v2_missingness_indicators",
    X_train=X_train_v2,
    y_train=dataset_v0.y_train.copy(),
    X_test=X_test_v2,
    metadata={
        "description": (
            "v0_raw_minimal with row-level missing counts "
            "and per-feature missing indicators."
        ),
        "source": "v0_raw_minimal",
        "base_version": "v0_raw_minimal",
        "summary_count_features": summary_count_columns_v2,
        "indicator_source_features": (
            missing_indicator_features_v2
        ),
        "indicator_features": indicator_columns_v2,
        "added_features": added_features_v2,
        "n_summary_count_features": (
            len(summary_count_columns_v2)
        ),
        "n_indicator_features": (
            len(indicator_columns_v2)
        ),
        "n_added_features": len(added_features_v2),
        "n_train_rows": len(X_train_v2),
        "n_test_rows": len(X_test_v2),
        "n_features": X_train_v2.shape[1],
    },
)

print(f"Dataset version:    {dataset_v2.version}")
print(f"Train shape:        {dataset_v2.X_train.shape}")
print(f"Test shape:         {dataset_v2.X_test.shape}")
print(
    "Summary features:  "
    f"{len(summary_count_columns_v2)}"
)
print(
    "Indicator features:"
    f" {len(indicator_columns_v2)}"
)
print(f"Total added:        {len(added_features_v2)}")

Dataset version:    v2_missingness_indicators
Train shape:        (10000, 393)
Test shape:         (2500, 393)
Summary features:  4
Indicator features: 184
Total added:        188


In [16]:
display(
    X_train_v2[
        summary_count_columns_v2
    ]
    .describe()
    .T
)

display(
    pd.DataFrame(
        {
            "indicator": indicator_columns_v2,
            "train_missing_rate": [
                X_train_v2[column].mean()
                for column in indicator_columns_v2
            ],
            "test_missing_rate": [
                X_test_v2[column].mean()
                for column in indicator_columns_v2
            ],
        }
    )
    .sort_values(
        "train_missing_rate",
        ascending=False,
    )
    .head(30)
)

,count,mean,std,min,25%,50%,75%,max
missing_count_total,10000.0,135.4677,4.947242,128.0,132.0,135.0,137.0,157.0
missing_count_numeric,10000.0,131.5308,3.168889,128.0,130.0,131.0,132.0,148.0
missing_count_categorical,10000.0,3.9369,2.850076,0.0,2.0,4.0,6.0,12.0
missing_count_very_high,10000.0,126.3432,8.201155,94.0,129.0,129.0,129.0,129.0


,indicator,train_missing_rate,test_missing_rate
167,Var190_is_missing,0.9957,0.9956
77,Var92_is_missing,0.9957,0.9948
51,Var64_is_missing,0.9954,0.9940
36,Var45_is_missing,0.9922,0.9908
87,Var102_is_missing,0.9918,0.9920
83,Var98_is_missing,0.9896,0.9864
9,Var12_is_missing,0.9896,0.9864
49,Var62_is_missing,0.9896,0.9864
74,Var89_is_missing,0.9869,0.9836
137,Var156_is_missing,0.9869,0.9848


In [17]:
saved_path_v2 = save_dataset(
    dataset=dataset_v2,
    processed_dir=PROCESSED_DIR,
    overwrite=True,
)

print(f"Saved to: {saved_path_v2}")

Saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\data\processed\v2_missingness_indicators


## Dataset Version v3: Targeted Missingness

This version is built on top of `v1_missingness_summary`.

It retains the eight row-level missingness summary features and adds binary missing-value indicators only for the source features that received meaningful importance in the broad `v2_missingness_indicators` experiment.

Selected source features:

- `Var217`
- `Var126`
- `Var218`
- `Var192`

This targeted approach tests whether the useful signal from individual missingness patterns can be preserved without adding the full set of noisy indicators.

In [18]:
targeted_indicator_features_v3 = [
    "Var217",
    "Var126",
    "Var218",
    "Var192",
]

missing_targeted_features = [
    column
    for column in targeted_indicator_features_v3
    if column not in X_train_v1.columns
]

if missing_targeted_features:
    raise KeyError(
        "Targeted source features are absent from v1: "
        f"{missing_targeted_features}"
    )

print(
    "Targeted missingness source features: "
    f"{len(targeted_indicator_features_v3)}"
)

print(*targeted_indicator_features_v3, sep="\n")

Targeted missingness source features: 4
Var217
Var126
Var218
Var192


In [19]:
X_train_v3 = add_selected_missing_indicators(
    X_train_v1,
    indicator_features=targeted_indicator_features_v3,
)

X_test_v3 = add_selected_missing_indicators(
    X_test_v1,
    indicator_features=targeted_indicator_features_v3,
)

indicator_columns_v3 = [
    f"{column}_is_missing"
    for column in targeted_indicator_features_v3
]

added_features_v3 = [
    column
    for column in X_train_v3.columns
    if column not in X_train_v1.columns
]

print(f"Added features: {added_features_v3}")

Added features: ['Var217_is_missing', 'Var126_is_missing', 'Var218_is_missing', 'Var192_is_missing']


In [20]:
if list(X_train_v3.columns) != list(X_test_v3.columns):
    raise ValueError(
        "Train and test columns differ after v3 feature generation."
    )

if X_train_v3.columns.duplicated().any():
    raise ValueError(
        "Generated train dataset contains duplicate columns."
    )

if X_test_v3.columns.duplicated().any():
    raise ValueError(
        "Generated test dataset contains duplicate columns."
    )

if added_features_v3 != indicator_columns_v3:
    raise ValueError(
        "Unexpected set or order of generated v3 features."
    )

if X_train_v3[indicator_columns_v3].isna().any().any():
    raise ValueError(
        "Train targeted indicators contain missing values."
    )

if X_test_v3[indicator_columns_v3].isna().any().any():
    raise ValueError(
        "Test targeted indicators contain missing values."
    )

train_indicator_values = set(
    X_train_v3[indicator_columns_v3]
    .to_numpy()
    .ravel()
)

test_indicator_values = set(
    X_test_v3[indicator_columns_v3]
    .to_numpy()
    .ravel()
)

if not train_indicator_values.issubset({0, 1}):
    raise ValueError(
        "Train indicators contain values other than 0 and 1."
    )

if not test_indicator_values.issubset({0, 1}):
    raise ValueError(
        "Test indicators contain values other than 0 and 1."
    )

pd.testing.assert_frame_equal(
    X_train_v3[X_train_v1.columns],
    X_train_v1,
)

pd.testing.assert_frame_equal(
    X_test_v3[X_test_v1.columns],
    X_test_v1,
)

print("v3 validation passed.")

v3 validation passed.


In [21]:
dataset_v3 = PreparedDataset(
    version="v3_targeted_missingness",
    X_train=X_train_v3,
    y_train=dataset_v1.y_train.copy(),
    X_test=X_test_v3,
    metadata={
        "description": (
            "v1_missingness_summary with targeted per-feature "
            "missing indicators selected from the broad v2 experiment."
        ),
        "source": "v1_missingness_summary",
        "base_version": "v1_missingness_summary",
        "selection_source_experiment": (
            "xgboost_v2_missingness_indicators_cv5"
        ),
        "indicator_source_features": (
            targeted_indicator_features_v3
        ),
        "indicator_features": indicator_columns_v3,
        "added_features": added_features_v3,
        "n_added_features": len(added_features_v3),
        "n_train_rows": len(X_train_v3),
        "n_test_rows": len(X_test_v3),
        "n_features": X_train_v3.shape[1],
    },
)

print(f"Dataset version: {dataset_v3.version}")
print(f"Train shape:     {dataset_v3.X_train.shape}")
print(f"Target shape:    {dataset_v3.y_train.shape}")
print(f"Test shape:      {dataset_v3.X_test.shape}")
print(f"Added features:  {len(added_features_v3)}")

Dataset version: v3_targeted_missingness
Train shape:     (10000, 217)
Target shape:    (10000,)
Test shape:      (2500, 217)
Added features:  4


In [22]:
indicator_summary_v3 = pd.DataFrame(
    {
        "indicator": indicator_columns_v3,
        "train_rate": [
            X_train_v3[column].mean()
            for column in indicator_columns_v3
        ],
        "test_rate": [
            X_test_v3[column].mean()
            for column in indicator_columns_v3
        ],
    }
)

indicator_summary_v3["absolute_difference"] = (
    indicator_summary_v3["train_rate"]
    - indicator_summary_v3["test_rate"]
).abs()

display(
    indicator_summary_v3.sort_values(
        "absolute_difference",
        ascending=False,
    )
)

,indicator,train_rate,test_rate,absolute_difference
1,Var126_is_missing,0.2780,0.2848,0.0068
0,Var217_is_missing,0.0128,0.0156,0.0028
2,Var218_is_missing,0.0128,0.0156,0.0028
3,Var192_is_missing,0.0079,0.0084,0.0005


In [23]:
saved_path_v3 = save_dataset(
    dataset=dataset_v3,
    processed_dir=PROCESSED_DIR,
    overwrite=True,
)

print(f"Saved to: {saved_path_v3}")

Saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\data\processed\v3_targeted_missingness
